<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M0_BRIDGE_2048_SOURCE_LOCK_SCHEMA_v1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M0_BRIDGE_2048_SOURCE_LOCK_SCHEMA_v1.2

**목적**: v1.1 결과를 재사용하고, 26/30으로 실패한 CSV schema 원인을 진단한 뒤
수정된 로더로 Learning 전체 8,384개 CSV를 확정한다.

---

## 재사용하는 기존 v1.1 결과

| 파일 | 용도 |
|:---|:---|
| `SOURCE_LOCK_SUMMARY.json` | 전체 카운트·Gate 결과 |
| `femto_csv_source_manifest.csv` | 43,369개 파일 목록 |
| `csv_structure_probe.csv` | 기존 30개 probe 결과 |
| `csv_duplicate_hash_within_record.csv` | within-record 중복 결과 |
| `source_file_immutability_check.csv` | 기준 파일 불변성 |
| `audit_gate_check.json` | 감사 Gate 결과 |

## 이번 단계에서 수행하지 않는 작업

- 전체 43,369개 fingerprint 재계산
- 전체 duplicate hash 재계산
- M0 감사 재실행
- M0 feature 재계산
- 원본 CSV 변경
- 2560→2048 변환

## PASS 기준 (허용)

| 상태 | 의미 |
|:---|:---|
| `PASS_NATIVE_2560` | 정확히 2560행 순수 numeric |
| `PASS_SINGLE_HEADER_REMOVED` | 첫 행만 비수치 header, 이후 2560행 |
| `PASS_SINGLE_FOOTER_REMOVED` | 마지막 행만 비수치 footer, 이전 2560행 |

## 절대 금지

- 2,561개 숫자 데이터 중 하나를 임의 제거
- 2,559개를 zero padding
- 부족한 데이터를 보간
- 초과 numeric 행을 crop
- 실패 파일을 제외하고 PASS 처리
- 원본 파일 수정

## 실행 순서

1. **셀 01** — 단위 검증 (드라이브 마운트 불필요)
2. **셀 02** — 메인 실행 (v1.1 재사용 + 전체 schema 확정)
3. **셀 03** — 결과 확인 (언제든 독립 실행 가능)

## 재실행 시

`_checkpoints/source_lock_schema_v1_2/learning_schema_checkpoint.csv`에
100개 단위로 저장됩니다. 재실행하면 완료된 파일을 건너뜁니다.

In [1]:
# ================================================================
# 셀 01 — 단위 검증
# ================================================================

import re
import numpy as np
import pandas as pd

# ── 기본 상수 ────────────────────────────────────────────────
assert int(25600 * 0.1) == 2560, "EXPECTED_SAMPLES 오류"
assert 4 == 4, "COL_H 오류"
assert 8384 == 8384, "EXPECTED_LEARNING_FILES 오류"
print("[Unit] 상수 assert 통과 ✅")

# ── natural sort ─────────────────────────────────────────────
def _nk(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(s))]

assert sorted(["acc_10.csv","acc_2.csv","acc_1.csv"], key=_nk) == \
       ["acc_1.csv","acc_2.csv","acc_10.csv"]
print("[Unit] natural sort 통과 ✅")

# ── extract_h_signal_strict 로직 검증 ────────────────────────
EXPECTED_SAMPLES_TEST = 2560
COL_H_TEST = 4

def _extract_test(col_values):
    """단위 테스트용 간소화 버전."""
    raw = pd.to_numeric(pd.Series(col_values), errors="coerce")
    n = len(raw)
    if n == EXPECTED_SAMPLES_TEST and raw.notna().all():
        return "PASS_NATIVE_2560"
    if n == EXPECTED_SAMPLES_TEST+1 and pd.isna(raw.iloc[0]) and raw.iloc[1:].notna().all():
        return "PASS_SINGLE_HEADER_REMOVED"
    if n == EXPECTED_SAMPLES_TEST+1 and raw.iloc[:-1].notna().all() and pd.isna(raw.iloc[-1]):
        return "PASS_SINGLE_FOOTER_REMOVED"
    return "FAIL_UNRESOLVED_SCHEMA"

# Case 1: 2560개 순수 numeric
assert _extract_test([1.0] * 2560) == "PASS_NATIVE_2560"
# Case 2: header 1행 + 2560개
assert _extract_test(["header"] + [1.0]*2560) == "PASS_SINGLE_HEADER_REMOVED"
# Case 3: 2560개 + footer 1행
assert _extract_test([1.0]*2560 + ["footer"]) == "PASS_SINGLE_FOOTER_REMOVED"
# Case 4: 2561개 숫자 → FAIL (crop 금지)
assert _extract_test([1.0] * 2561) == "FAIL_UNRESOLVED_SCHEMA"
# Case 5: 2559개 → FAIL (padding 금지)
assert _extract_test([1.0] * 2559) == "FAIL_UNRESOLVED_SCHEMA"
print("[Unit] extract_h_signal_strict 로직 통과 ✅")

print("[Unit] 모든 단위 검증 통과 ✅")

[Unit] 상수 assert 통과 ✅
[Unit] natural sort 통과 ✅
[Unit] extract_h_signal_strict 로직 통과 ✅
[Unit] 모든 단위 검증 통과 ✅


In [2]:
# ================================================================
# 셀 02 — 메인 실행
# ================================================================

# ================================================================
# M0_BRIDGE_2048_SOURCE_LOCK_SCHEMA_v1.2
# 기존 v1.1 결과 재사용 + Learning 8,384개 schema 확정
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import csv
import hashlib
import json
import os
import re
import traceback

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


# ================================================================
# 1. 설정
# ================================================================

VERSION = "M0_BRIDGE_2048_SOURCE_LOCK_SCHEMA_v1.2"

PROJECT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
    Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
]

PROJECT_ROOT = None
for _c in PROJECT_ROOT_CANDIDATES:
    if _c.exists():
        PROJECT_ROOT = _c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("PROJECT_ROOT_NOT_FOUND")

EXPECTED_LEARNING_FILES = 8384
EXPECTED_SAMPLES = 2560
COL_H = 4

RUN_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S_source_lock_schema_v1_2"
)

OUTPUT_DIR = PROJECT_ROOT / "bridge_outputs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

CHECKPOINT_DIR = (
    PROJECT_ROOT / "bridge_outputs"
    / "_checkpoints" / "source_lock_schema_v1_2"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_CHECKPOINT = CHECKPOINT_DIR / "learning_schema_checkpoint.csv"

print("VERSION    :", VERSION)
print("OUTPUT_DIR :", OUTPUT_DIR)


# ================================================================
# 2. 공통 함수
# ================================================================

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as file:
        json.dump(obj, file, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)


def natural_key(value):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", str(value))
    ]


def safe_read_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def safe_read_optional_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def detect_separator(path, encoding):
    with Path(path).open("r", encoding=encoding, errors="replace") as file:
        sample = file.read(8192)
    candidates = [",", ";", "\t"]
    counts = {sep: sample.count(sep) for sep in candidates}
    sep = max(counts, key=counts.get)
    return sep if counts[sep] > 0 else ","


def read_frame_flexible(path):
    """
    delimiter 및 encoding 변형을 진단한다.
    원본 파일은 변경하지 않는다.
    """
    path = Path(path)
    attempts = []
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr", "latin-1"]
    last_error = None

    for encoding in encodings:
        try:
            sep = detect_separator(path, encoding)
            frame = pd.read_csv(
                path, header=None, sep=sep,
                encoding=encoding, engine="python",
                on_bad_lines="error",
            )
            attempts.append({
                "encoding": encoding, "separator": repr(sep),
                "status": "SUCCESS",
                "rows": len(frame), "columns": frame.shape[1],
            })
            return frame, encoding, sep, attempts
        except Exception as error:
            last_error = error
            attempts.append({
                "encoding": encoding, "separator": None,
                "status": "FAIL", "error": repr(error),
            })

    raise RuntimeError(f"CSV_PARSE_FAILED: {last_error}")


def extract_h_signal_strict(frame):
    """
    허용:
      1) 2560개 순수 데이터              → PASS_NATIVE_2560
      2) 첫 행 비수치 header + 2560개    → PASS_SINGLE_HEADER_REMOVED
      3) 마지막 행 비수치 footer + 2560개 → PASS_SINGLE_FOOTER_REMOVED

    금지: numeric row 임의 제거 / padding / crop / interpolation
    """
    if frame.shape[1] <= COL_H:
        return {"status": "FAIL_COLUMN_MISSING", "signal": None,
                "data_rows": 0, "header_rows_removed": 0, "footer_rows_removed": 0}

    raw = pd.to_numeric(frame.iloc[:, COL_H], errors="coerce")
    n = len(raw)

    # Case 1: 정확히 2560개 순수 numeric
    if n == EXPECTED_SAMPLES and raw.notna().all():
        sig = raw.to_numpy(dtype=float)
        if np.isfinite(sig).all():
            return {"status": "PASS_NATIVE_2560", "signal": sig,
                    "data_rows": EXPECTED_SAMPLES,
                    "header_rows_removed": 0, "footer_rows_removed": 0}

    # Case 2: 첫 행만 비수치 header
    if (n == EXPECTED_SAMPLES + 1
            and pd.isna(raw.iloc[0])
            and raw.iloc[1:].notna().all()):
        sig = raw.iloc[1:].to_numpy(dtype=float)
        if np.isfinite(sig).all():
            return {"status": "PASS_SINGLE_HEADER_REMOVED", "signal": sig,
                    "data_rows": EXPECTED_SAMPLES,
                    "header_rows_removed": 1, "footer_rows_removed": 0}

    # Case 3: 마지막 행만 비수치 footer
    if (n == EXPECTED_SAMPLES + 1
            and raw.iloc[:-1].notna().all()
            and pd.isna(raw.iloc[-1])):
        sig = raw.iloc[:-1].to_numpy(dtype=float)
        if np.isfinite(sig).all():
            return {"status": "PASS_SINGLE_FOOTER_REMOVED", "signal": sig,
                    "data_rows": EXPECTED_SAMPLES,
                    "header_rows_removed": 0, "footer_rows_removed": 1}

    numeric_count = int(raw.notna().sum())
    nan_count = int(raw.isna().sum())
    return {"status": "FAIL_UNRESOLVED_SCHEMA", "signal": None,
            "data_rows": numeric_count, "raw_rows": n,
            "nan_count": nan_count,
            "header_rows_removed": 0, "footer_rows_removed": 0}


def inspect_raw_lines(path, count=3):
    result = {"head_lines": [], "tail_lines": []}
    try:
        with Path(path).open("r", encoding="utf-8-sig", errors="replace") as file:
            lines = file.readlines()
        result["raw_line_count"] = len(lines)
        result["head_lines"] = [repr(l[:500]) for l in lines[:count]]
        result["tail_lines"] = [repr(l[:500]) for l in lines[-count:]]
    except Exception as error:
        result["raw_inspection_error"] = repr(error)
    return result


# ================================================================
# 3. 기존 v1.1 결과 자동 탐색
# ================================================================

v1_candidates = sorted(
    [
        path for path in (PROJECT_ROOT / "bridge_outputs").glob("*_source_lock_v1_1")
        if path.is_dir()
        and (path / "SOURCE_LOCK_SUMMARY.json").exists()
    ],
    key=lambda path: path.stat().st_mtime_ns,
    reverse=True,
)

if not v1_candidates:
    raise FileNotFoundError("SOURCE_LOCK_V1_1_RESULT_NOT_FOUND")

V1_DIR = v1_candidates[0]

print("V1_DIR    :", V1_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

v1_summary    = safe_read_json(V1_DIR / "SOURCE_LOCK_SUMMARY.json")
manifest      = pd.read_csv(V1_DIR / "femto_csv_source_manifest.csv")
old_probe     = pd.read_csv(V1_DIR / "csv_structure_probe.csv")
within_duplicates = safe_read_optional_csv(V1_DIR / "csv_duplicate_hash_within_record.csv")
immutability  = pd.read_csv(V1_DIR / "source_file_immutability_check.csv")
audit_gate    = safe_read_json(V1_DIR / "audit_gate_check.json")

print(f"Manifest rows   : {len(manifest):,}")
print(f"Old probe rows  : {len(old_probe)}")
print(f"Within dup rows : {len(within_duplicates)}")


# ================================================================
# 4. 기존 4개 실패 파일 상세 진단
# ================================================================

failed_probe = old_probe[old_probe["probe_status"] != "PASS"].copy()

print(f"\n[DIAG] 기존 probe 실패: {len(failed_probe)}개")

diagnostic_rows = []

for row in failed_probe.to_dict(orient="records"):
    path = Path(row["absolute_path"])

    result = {
        "record_uid":             row.get("record_uid"),
        "probe_position":         row.get("probe_position"),
        "sequence_index":         row.get("sequence_index"),
        "csv_file_name":          row.get("csv_file_name"),
        "absolute_path":          str(path),
        "previous_probe_status":  row.get("probe_status"),
        "previous_error_message": row.get("error_message"),
    }

    try:
        frame, encoding, sep, attempts = read_frame_flexible(path)
        extraction = extract_h_signal_strict(frame)
        raw_info   = inspect_raw_lines(path)

        result.update({
            "detected_encoding":    encoding,
            "detected_separator":   repr(sep),
            "raw_row_count":        len(frame),
            "column_count":         frame.shape[1],
            "corrected_status":     extraction["status"],
            "corrected_data_rows":  extraction.get("data_rows"),
            "header_rows_removed":  extraction.get("header_rows_removed", 0),
            "footer_rows_removed":  extraction.get("footer_rows_removed", 0),
            "read_attempts":        json.dumps(attempts, ensure_ascii=False),
            **raw_info,
        })
    except Exception as error:
        result.update({
            "corrected_status": "FAIL_PARSE_EXCEPTION",
            "corrected_error":  repr(error),
            **inspect_raw_lines(path),
        })

    diagnostic_rows.append(result)

DIAGNOSTIC_DF = pd.DataFrame(diagnostic_rows)
DIAGNOSTIC_DF.to_csv(
    OUTPUT_DIR / "failed_probe_diagnostics.csv",
    index=False, encoding="utf-8-sig"
)

print("\n[DIAG] 실패 파일 진단 결과:")
disp_cols = [c for c in [
    "record_uid","probe_position","csv_file_name",
    "previous_probe_status","corrected_status",
    "raw_row_count","column_count",
    "corrected_data_rows","header_rows_removed","footer_rows_removed",
] if c in DIAGNOSTIC_DF.columns]
display(DIAGNOSTIC_DF[disp_cols])


# ================================================================
# 5. Learning 8,384개 전체 schema 검사
# ================================================================

learning = (
    manifest[manifest["split"] == "LEARNING"]
    .sort_values(["record_uid", "sequence_index"])
    .reset_index(drop=True)
)

if len(learning) != EXPECTED_LEARNING_FILES:
    raise RuntimeError(
        f"LEARNING_MANIFEST_COUNT_MISMATCH: "
        f"{len(learning)} != {EXPECTED_LEARNING_FILES}"
    )

# 체크포인트 로드
checkpoint_cache = {}
if SCHEMA_CHECKPOINT.exists():
    try:
        cached_df = pd.read_csv(SCHEMA_CHECKPOINT)
        for row in cached_df.to_dict(orient="records"):
            key = (
                str(row["absolute_path"]),
                int(float(row["file_size_bytes"])),
                int(float(row["mtime_ns"])),
            )
            checkpoint_cache[key] = row
        print(f"[SCHEMA] 체크포인트 로드: {len(checkpoint_cache):,}개")
    except Exception as error:
        print(f"[SCHEMA] 체크포인트 로드 실패 (무시): {error}")
        checkpoint_cache = {}

schema_rows = []

for index, row in enumerate(
    learning.to_dict(orient="records"),
    start=1,
):
    key = (
        str(row["absolute_path"]),
        int(row["file_size_bytes"]),
        int(row["mtime_ns"]),
    )

    cached = checkpoint_cache.get(key)
    if cached is not None:
        schema_rows.append(cached)
        continue

    result = {
        "record_uid":         row["record_uid"],
        "logical_bearing_id": row["logical_bearing_id"],
        "sequence_index":     int(row["sequence_index"]),
        "csv_file_name":      row["csv_file_name"],
        "absolute_path":      row["absolute_path"],
        "file_size_bytes":    int(row["file_size_bytes"]),
        "mtime_ns":           int(row["mtime_ns"]),
        "schema_status":      "FAIL_NOT_EXECUTED",
        "error_message":      None,
    }

    try:
        frame, encoding, sep, attempts = read_frame_flexible(row["absolute_path"])
        extraction = extract_h_signal_strict(frame)
        signal = extraction.get("signal")

        if signal is not None:
            constant_signal = bool(np.std(signal) == 0)
            finite_pass     = bool(np.isfinite(signal).all())
            schema_pass = (
                len(signal) == EXPECTED_SAMPLES
                and finite_pass
                and not constant_signal
            )
            result.update({
                "encoding":            encoding,
                "separator":           repr(sep),
                "raw_row_count":       len(frame),
                "column_count":        int(frame.shape[1]),
                "data_row_count":      len(signal),
                "header_rows_removed": extraction.get("header_rows_removed", 0),
                "footer_rows_removed": extraction.get("footer_rows_removed", 0),
                "nan_count":           int(np.isnan(signal).sum()),
                "inf_count":           int(np.isinf(signal).sum()),
                "constant_signal":     constant_signal,
                "rms":                 float(np.sqrt(np.mean(signal ** 2))),
                "schema_status":       extraction["status"]
                                       if schema_pass
                                       else "FAIL_SIGNAL_QUALITY",
            })
        else:
            result.update({
                "encoding":        encoding,
                "separator":       repr(sep),
                "raw_row_count":   len(frame),
                "column_count":    int(frame.shape[1]),
                "data_row_count":  extraction.get("data_rows"),
                "schema_status":   extraction["status"],
            })

    except Exception as error:
        result.update({
            "schema_status": "FAIL_PARSE_EXCEPTION",
            "error_message": repr(error),
        })

    schema_rows.append(result)
    checkpoint_cache[key] = result

    if index % 100 == 0:
        chk_df = pd.DataFrame(list(checkpoint_cache.values()))
        tmp = SCHEMA_CHECKPOINT.with_suffix(".csv.tmp")
        chk_df.to_csv(tmp, index=False, encoding="utf-8-sig")
        os.replace(tmp, SCHEMA_CHECKPOINT)
        print(f"[SCHEMA] {index:,}/{EXPECTED_LEARNING_FILES:,}")

# 최종 체크포인트 저장
final_chk_df = pd.DataFrame(list(checkpoint_cache.values()))
final_chk_df.to_csv(SCHEMA_CHECKPOINT, index=False, encoding="utf-8-sig")

SCHEMA_DF = pd.DataFrame(schema_rows)
SCHEMA_DF.to_csv(
    OUTPUT_DIR / "learning_schema_full.csv",
    index=False, encoding="utf-8-sig"
)

fail_mask = ~SCHEMA_DF["schema_status"].astype(str).str.startswith("PASS", na=False)
SCHEMA_DF[fail_mask].to_csv(
    OUTPUT_DIR / "learning_schema_failures.csv",
    index=False, encoding="utf-8-sig"
)

print(f"[SCHEMA] 완료: {len(SCHEMA_DF):,}개 검사")


# ================================================================
# 6. 최종 판정
# ================================================================

schema_pass_mask    = SCHEMA_DF["schema_status"].astype(str).str.startswith("PASS", na=False)
schema_pass_count   = int(schema_pass_mask.sum())
schema_failure_count = int((~schema_pass_mask).sum())

status_counts = (
    SCHEMA_DF["schema_status"].value_counts(dropna=False).to_dict()
)

within_duplicate_count = len(within_duplicates)

immutability_pass = bool(
    not immutability.empty
    and immutability["unchanged"].all()
)

audit_gate_pass = bool(audit_gate.get("gate_pass") is True)

v1_total = (
    v1_summary
    .get("current_complete_dataset", {})
    .get("actual_total")
)

count_pass = (
    v1_total == 43369
    and len(learning) == EXPECTED_LEARNING_FILES
)

if not audit_gate_pass:
    final_status = "SOURCE_LOCK_BLOCKED_BY_AUDIT_GATE"
    next_action  = "FIX_AUDIT_GATE_FIRST"
elif not count_pass:
    final_status = "SOURCE_LOCK_HOLD_COUNT_MISMATCH"
    next_action  = "REVIEW_SOURCE_INVENTORY"
elif within_duplicate_count > 0:
    final_status = "SOURCE_LOCK_HOLD_DUPLICATE_CONTENT"
    next_action  = "MANUAL_DUPLICATE_REVIEW"
elif schema_failure_count > 0:
    final_status = "SOURCE_LOCK_HOLD_SCHEMA_MISMATCH"
    next_action  = "REVIEW_SCHEMA_FAILURE_FILES"
elif not immutability_pass:
    final_status = "SOURCE_LOCK_INVALID_SOURCE_MODIFIED"
    next_action  = "MANUAL_REVIEW_REQUIRED"
else:
    final_status = "SOURCE_LOCK_PASS"
    next_action  = "READY_FOR_M0_BRIDGE_2048_PREP"

summary = {
    "version":                        VERSION,
    "created_at":                      datetime.now().isoformat(timespec="seconds"),
    "base_source_lock_directory":      str(V1_DIR),
    "source_lock_status":              final_status,
    "recommended_next_action":         next_action,
    "audit_gate_pass":                 audit_gate_pass,
    "dataset_total_csv":               v1_total,
    "learning_expected":               EXPECTED_LEARNING_FILES,
    "learning_actual":                 len(learning),
    "learning_schema_pass_count":      schema_pass_count,
    "learning_schema_failure_count":   schema_failure_count,
    "schema_status_counts":            status_counts,
    "previous_probe_failure_count":    len(failed_probe),
    "within_record_duplicate_groups":  within_duplicate_count,
    "source_files_unchanged":          immutability_pass,
    "numeric_rows_truncated":          0,
    "padding_performed":               False,
    "interpolation_performed":         False,
    "original_files_modified":         False,
    "bridge_conversion_performed":     False,
    "output_directory":                str(OUTPUT_DIR),
}

write_json(OUTPUT_DIR / "SOURCE_LOCK_SCHEMA_SUMMARY.json", summary)

summary_md = f"""# Source Lock Schema v1.2

- **Status:** `{final_status}`
- **Next action:** `{next_action}`
- **Dataset CSV:** `{v1_total:,}`
- **Learning CSV:** `{len(learning):,}`
- **Learning schema pass:** `{schema_pass_count:,}`
- **Learning schema failure:** `{schema_failure_count:,}`
- **Schema status counts:** `{status_counts}`
- **Previous probe failures:** `{len(failed_probe)}`
- **Within-record duplicates:** `{within_duplicate_count}`
- **Source unchanged:** `{immutability_pass}`
- **Numeric truncation:** `0`
- **Padding:** `False`
- **Interpolation:** `False`
- **Output:** `{OUTPUT_DIR}`
"""

with (OUTPUT_DIR / "SOURCE_LOCK_SCHEMA_SUMMARY.md").open("w", encoding="utf-8") as file:
    file.write(summary_md)

print("\n" + "=" * 72)
print("SOURCE LOCK STATUS  :", final_status)
print("NEXT ACTION         :", next_action)
print(f"LEARNING SCHEMA     : {schema_pass_count:,}/{EXPECTED_LEARNING_FILES:,}")
print("SCHEMA STATUS CNT   :", status_counts)
print("SCHEMA FAILURES     :", schema_failure_count)
print("WITHIN DUPLICATES   :", within_duplicate_count)
print("SOURCE UNCHANGED    :", immutability_pass)
print("OUTPUT DIR          :", OUTPUT_DIR)
print("=" * 72)

Mounted at /content/drive
VERSION    : M0_BRIDGE_2048_SOURCE_LOCK_SCHEMA_v1.2
OUTPUT_DIR : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_155856_source_lock_schema_v1_2
V1_DIR    : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_150815_source_lock_v1_1
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_155856_source_lock_schema_v1_2
Manifest rows   : 43,369
Old probe rows  : 30
Within dup rows : 0

[DIAG] 기존 probe 실패: 4개

[DIAG] 실패 파일 진단 결과:


,record_uid,probe_position,csv_file_name,previous_probe_status,corrected_status,raw_row_count,column_count,corrected_data_rows,header_rows_removed,footer_rows_removed
0,LEARNING/Bearing1_1,LAST,temp_00466.csv,FAIL,FAIL_UNRESOLVED_SCHEMA,517,5,517,0,0
1,LEARNING/Bearing1_2,LAST,temp_00144.csv,ERROR,FAIL_UNRESOLVED_SCHEMA,596,5,596,0,0
2,LEARNING/Bearing2_1,LAST,temp_00151.csv,ERROR,FAIL_UNRESOLVED_SCHEMA,37,5,37,0,0
3,LEARNING/Bearing3_1,LAST,temp_00089.csv,ERROR,FAIL_UNRESOLVED_SCHEMA,280,5,280,0,0


[SCHEMA] 100/8,384
[SCHEMA] 200/8,384
[SCHEMA] 300/8,384
[SCHEMA] 400/8,384
[SCHEMA] 500/8,384
[SCHEMA] 600/8,384
[SCHEMA] 700/8,384
[SCHEMA] 800/8,384
[SCHEMA] 900/8,384
[SCHEMA] 1,000/8,384
[SCHEMA] 1,100/8,384
[SCHEMA] 1,200/8,384
[SCHEMA] 1,300/8,384
[SCHEMA] 1,400/8,384
[SCHEMA] 1,500/8,384
[SCHEMA] 1,600/8,384
[SCHEMA] 1,700/8,384
[SCHEMA] 1,800/8,384
[SCHEMA] 1,900/8,384
[SCHEMA] 2,000/8,384
[SCHEMA] 2,100/8,384
[SCHEMA] 2,200/8,384
[SCHEMA] 2,300/8,384
[SCHEMA] 2,400/8,384
[SCHEMA] 2,500/8,384
[SCHEMA] 2,600/8,384
[SCHEMA] 2,700/8,384
[SCHEMA] 2,800/8,384
[SCHEMA] 2,900/8,384
[SCHEMA] 3,000/8,384
[SCHEMA] 3,100/8,384
[SCHEMA] 3,200/8,384
[SCHEMA] 3,300/8,384
[SCHEMA] 3,400/8,384
[SCHEMA] 3,500/8,384
[SCHEMA] 3,600/8,384
[SCHEMA] 3,700/8,384
[SCHEMA] 3,800/8,384
[SCHEMA] 3,900/8,384
[SCHEMA] 4,000/8,384
[SCHEMA] 4,100/8,384
[SCHEMA] 4,200/8,384
[SCHEMA] 4,300/8,384
[SCHEMA] 4,400/8,384
[SCHEMA] 4,500/8,384
[SCHEMA] 4,600/8,384
[SCHEMA] 4,700/8,384
[SCHEMA] 4,800/8,384
[SCHEMA] 4

In [3]:
# ================================================================
# 셀 03 — 결과 확인 (언제든 독립 실행 가능)
# ================================================================

import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT_CANDIDATES_C = [
    Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
    Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
]
_pr = None
for _c in PROJECT_ROOT_CANDIDATES_C:
    if _c.exists():
        _pr = _c
        break

if _pr is None:
    print("PROJECT_ROOT 없음 — 드라이브 마운트 후 실행")
else:
    # 최신 schema_v1_2 폴더 탐색
    schema_dirs = sorted(
        [p for p in (_pr / "bridge_outputs").glob("*_source_lock_schema_v1_2")
         if p.is_dir() and (p / "SOURCE_LOCK_SCHEMA_SUMMARY.json").exists()],
        key=lambda p: p.stat().st_mtime_ns, reverse=True,
    )

    if not schema_dirs:
        print("SOURCE_LOCK_SCHEMA_SUMMARY.json 없음 — 셀 02를 먼저 실행하세요")
    else:
        _sd = schema_dirs[0]
        with (_sd / "SOURCE_LOCK_SCHEMA_SUMMARY.json").open("r", encoding="utf-8") as f:
            s = json.load(f)

        print("=" * 60)
        print(f"SOURCE_LOCK_STATUS  : {s.get('source_lock_status')}")
        print(f"NEXT ACTION         : {s.get('recommended_next_action')}")
        print(f"Learning schema     : {s.get('learning_schema_pass_count')}/{s.get('learning_expected')}")
        print(f"Schema failures     : {s.get('learning_schema_failure_count')}")
        print(f"Schema status cnt   : {s.get('schema_status_counts')}")
        print(f"Within duplicates   : {s.get('within_record_duplicate_groups')}")
        print(f"Source unchanged    : {s.get('source_files_unchanged')}")
        print(f"Output dir          : {s.get('output_directory')}")
        print("=" * 60)

        # 실패 파일 상세 표시
        fail_path = _sd / "learning_schema_failures.csv"
        diag_path = _sd / "failed_probe_diagnostics.csv"

        if fail_path.exists():
            fail_df = pd.read_csv(fail_path)
            if len(fail_df) > 0:
                print(f"\n실패 파일 ({len(fail_df)}개):")
                disp = [c for c in [
                    "record_uid","csv_file_name","schema_status",
                    "raw_row_count","column_count","data_row_count",
                    "error_message"
                ] if c in fail_df.columns]
                display(fail_df[disp])
            else:
                print("\n실패 파일: 없음 ✅")

        if diag_path.exists():
            diag_df = pd.read_csv(diag_path)
            if len(diag_df) > 0:
                print(f"\n기존 실패 진단 ({len(diag_df)}개):")
                disp2 = [c for c in [
                    "record_uid","probe_position","csv_file_name",
                    "previous_probe_status","corrected_status",
                    "raw_row_count","column_count",
                    "corrected_data_rows","header_rows_removed","footer_rows_removed",
                ] if c in diag_df.columns]
                display(diag_df[disp2])

SOURCE_LOCK_STATUS  : SOURCE_LOCK_HOLD_SCHEMA_MISMATCH
NEXT ACTION         : REVIEW_SCHEMA_FAILURE_FILES
Learning schema     : 7534/8384
Schema failures     : 850
Schema status cnt   : {'PASS_NATIVE_2560': 7534, 'FAIL_UNRESOLVED_SCHEMA': 850}
Within duplicates   : 0
Source unchanged    : True
Output dir          : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_155856_source_lock_schema_v1_2

실패 파일 (850개):


,record_uid,csv_file_name,schema_status,raw_row_count,column_count,data_row_count,error_message
0,LEARNING/Bearing1_1,temp_00001.csv,FAIL_UNRESOLVED_SCHEMA,600,5,600,NaN
1,LEARNING/Bearing1_1,temp_00002.csv,FAIL_UNRESOLVED_SCHEMA,600,5,600,NaN
2,LEARNING/Bearing1_1,temp_00003.csv,FAIL_UNRESOLVED_SCHEMA,600,5,600,NaN
3,LEARNING/Bearing1_1,temp_00004.csv,FAIL_UNRESOLVED_SCHEMA,600,5,600,NaN
4,LEARNING/Bearing1_1,temp_00005.csv,FAIL_UNRESOLVED_SCHEMA,600,5,600,NaN
...,...,...,...,...,...,...,...
845,LEARNING/Bearing3_1,temp_00085.csv,FAIL_UNRESOLVED_SCHEMA,417,5,417,NaN
846,LEARNING/Bearing3_1,temp_00086.csv,FAIL_UNRESOLVED_SCHEMA,600,5,600,NaN
847,LEARNING/Bearing3_1,temp_00087.csv,FAIL_UNRESOLVED_SCHEMA,600,5,600,NaN
848,LEARNING/Bearing3_1,temp_00088.csv,FAIL_UNRESOLVED_SCHEMA,600,5,600,NaN



기존 실패 진단 (4개):


,record_uid,probe_position,csv_file_name,previous_probe_status,corrected_status,raw_row_count,column_count,corrected_data_rows,header_rows_removed,footer_rows_removed
0,LEARNING/Bearing1_1,LAST,temp_00466.csv,FAIL,FAIL_UNRESOLVED_SCHEMA,517,5,517,0,0
1,LEARNING/Bearing1_2,LAST,temp_00144.csv,ERROR,FAIL_UNRESOLVED_SCHEMA,596,5,596,0,0
2,LEARNING/Bearing2_1,LAST,temp_00151.csv,ERROR,FAIL_UNRESOLVED_SCHEMA,37,5,37,0,0
3,LEARNING/Bearing3_1,LAST,temp_00089.csv,ERROR,FAIL_UNRESOLVED_SCHEMA,280,5,280,0,0
